# Pino · Lectura de negocio por nivel
Complemento del notebook principal (§5 y §10). Usa el modelo y los datos que ese notebook guarda, por lo que se ejecuta en menos de un minuto.

**Preguntas:** ¿cuánto aporta el modelo frente a no usar modelo?, ¿qué pasa en los niveles infantiles? y ¿se puede mejorar sin reentrenar?

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_predict, KFold

SEED = 2026
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORTES = [0, 4, 6, 8, 10, 99]
NIVELES = ["N1", "N2", "N3", "N4", "Fuera de foco"]
nivel = lambda a, c=CORTES: pd.cut(pd.Series(np.asarray(a)), c, labels=False, right=False).to_numpy()
VARS = ["zipf", "n_letras", "n_fonemas", "n_silabas", "opacidad", "grupo_cons_max", "vecinos_ort", "concrecion", "categoria"]

en = pd.read_csv(RAIZ / "data" / "processed" / "palabras_en_modelado.csv")
X_train, X_test, y_train, y_test = train_test_split(en[VARS], en["aoa"], test_size=.2, stratify=nivel(en["aoa"]), random_state=SEED)
modelo = joblib.load(RAIZ / "models" / "modelo_aoa_en.joblib")
pred = modelo.predict(X_test); real = nivel(y_test)
print(f"MAE en test: {np.mean(np.abs(pred - y_test)):.4f} años (debe coincidir con el notebook principal: 1,5014)")

MAE en test: 1.5014 años (debe coincidir con el notebook principal: 1,5014)


## 1. Frente a no usar modelo

In [2]:
def metricas(asignado):
    infantil = real < 4
    return {"nivel exacto": np.mean(asignado == real), "nivel ±1": np.mean(np.abs(asignado - real) <= 1),
            "error grave": np.mean((y_test.to_numpy() >= 10) & (asignado < 2)),
            "infantiles: exacto": np.mean(asignado[infantil] == real[infantil]),
            "infantiles: ±1": np.mean(np.abs(asignado - real)[infantil] <= 1),
            "recall medio por nivel": np.mean([np.mean(asignado[real == k] == k) for k in range(5)])}
base = np.full(len(real), nivel([y_train.mean()])[0])
fijos = nivel(pred)
pd.DataFrame({"Sin modelo (siempre la media)": metricas(base), "Gradient Boosting": metricas(fijos)}).round(4)

,Sin modelo (siempre la media),Gradient Boosting
nivel exacto,0.6167,0.6742
nivel ±1,0.8158,0.9435
error grave,0.0000,0.0005
infantiles: exacto,0.0000,0.3525
infantiles: ±1,0.5195,0.8738
recall medio por nivel,0.2000,0.3886


**Lectura:** el 62 % de las palabras es "fuera de foco", así que **no usar modelo** ya logra 61,7 % de nivel exacto, 81,6 % de ±1 y cero errores graves. El aporte real en esas métricas es de +5,8 y +12,8 puntos. La diferencia grande está en las palabras infantiles, donde la línea base no acierta ninguna y el modelo acierta el 35 % (87 % a ±1, **bajo la meta de 90 %** si se mide solo en ellas).

## 2. Desempeño por nivel

In [3]:
def por_nivel(asignado):
    return pd.DataFrame([{"nivel": NIVELES[k], "palabras": int((real == k).sum()), "asignadas": int((asignado == k).sum()),
        "recall": np.mean(asignado[real == k] == k),
        "precisión": np.mean(real[asignado == k] == k) if (asignado == k).any() else np.nan} for k in range(5)]).set_index("nivel")
por_nivel(fijos).round(3)

,palabras,asignadas,recall,precisión
nivel,,,,
N1,64,1,0.016,1.000
N2,334,215,0.332,0.516
N3,664,513,0.331,0.429
N4,1148,1183,0.389,0.378
Fuera de foco,3555,3853,0.874,0.807


**Lectura:** por la regresión hacia la media, el modelo casi nunca asigna el nivel 1 (acierta 1 de 64 palabras).

## 3. Mejora: cortes calibrados

En lugar de cortar la predicción en 4, 6, 8 y 10 años, se eligen cortes para que la **proporción** de palabras por nivel sea la real. Se calculan solo con predicciones *out-of-fold* del **train**; el modelo no cambia. La idea surgió al revisar la matriz de test, por lo que el resultado en test es algo optimista y conviene confirmarlo con palabras nuevas.

In [4]:
oof = cross_val_predict(clone(modelo), X_train, y_train, cv=KFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
calibrados = [0] + [float(np.quantile(oof, np.mean(y_train < c))) for c in CORTES[1:-1]] + [99]
print("Cortes sobre la predicción:", [round(c, 2) for c in calibrados[1:-1]])
cal = nivel(pred, calibrados)
display(pd.DataFrame({"Cortes fijos": metricas(fijos), "Cortes calibrados": metricas(cal)}).round(4))
por_nivel(cal).round(3)

Cortes sobre la predicción: [4.84, 6.94, 8.82, 10.33]


,Cortes fijos,Cortes calibrados
nivel exacto,0.6742,0.6524
nivel ±1,0.9435,0.9417
error grave,0.0005,0.0014
infantiles: exacto,0.3525,0.3701
infantiles: ±1,0.8738,0.8995
recall medio por nivel,0.3886,0.4708


,palabras,asignadas,recall,precisión
nivel,,,,
N1,64,55,0.375,0.436
N2,334,346,0.440,0.425
N3,664,676,0.349,0.343
N4,1148,1135,0.361,0.366
Fuera de foco,3555,3553,0.828,0.828


## 4. Decisión

| | Cortes fijos | Cortes calibrados |
|---|---|---|
| Nivel exacto | **67,4 %** | 65,2 % |
| Recall del nivel 1 | 1,6 % | **37,5 %** |
| Palabras infantiles a ±1 | 87,4 % | **89,95 %** |
| Error grave (meta ≤ 2 %) | **0,05 %** | 0,14 % |

**Recomendación:** usar cortes calibrados para **proponer** niveles a las educadoras, porque recupera palabras de los primeros niveles a cambio de 2 puntos de nivel exacto y algo más de error grave, que sigue muy bajo la meta. La app y el dataset entregado siguen con los cortes fijos hasta validarlo. Incluso calibrado, el nivel 1 necesita datos de niños (Wordbank).